## Подготовка

In [8]:
# Автоперезагрузка модулей из src при их изменении
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# Функции из src
from src.data import load_data, save_submission, build_preprocessor
from src.validation import cross_validate, log_result
from src.models import build_model
from src.features import add_features

import warnings
warnings.filterwarnings("ignore", message="Found unknown categories")

In [50]:
# Загрузка конфига и данных
config = OmegaConf.load("config.yaml")
X, y, X_test, test_ids = load_data(config)

In [32]:
# Хелпер: собрать модель из конфига, прогнать CV и записать результат в журнал
def run_model(cfg, name=None):
    model = build_model(cfg)
    scores, _ = cross_validate(model, X, y, X_test, cfg)
    features = list(cfg.features.numeric) + list(cfg.features.categorical) + list(cfg.features.log_numeric)
    params = OmegaConf.to_container(cfg.model.params, resolve=True)
    log_result(cfg, name or cfg.model.name, params, features, scores)
    return float(round(scores.mean(), 4)), float(round(scores.std(), 4))

## Бейзлайн

In [7]:
# 3.2 Бейзлайн: всем домам медианная цена
from sklearn.dummy import DummyRegressor

scores, _ = cross_validate(DummyRegressor(strategy="median"), X, y, X_test, config)
log_result(config, "median", {}, [], scores)
round(scores.mean(), 4), round(scores.std(), 4)

(np.float64(0.3998), np.float64(0.0118))

In [8]:
# 3.3 Линейная регрессия на 4 сильнейших признаках: журнал и сабмит
print(run_model(config))
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "baseline_linreg_4features")

(np.float64(0.1842), np.float64(0.0296))


'outputs\\submission_baseline_linreg_4features.csv'

## Предобработка

In [7]:
# 4.1 Все 79 признаков как есть, Ridge(alpha=10): точка отсчёта
print(run_model(config))

(np.float64(0.1445), np.float64(0.0425))


In [12]:
# 4.2 Проверка: какие пропуски остались
print(X.isna().sum()[X.isna().sum() > 0])
print(X_test.isna().sum()[X_test.isna().sum() > 0])

LotFrontage    259
Electrical       1
GarageYrBlt     81
dtype: int64
MSZoning         4
LotFrontage    227
Utilities        2
Exterior1st      1
Exterior2nd      1
KitchenQual      1
Functional       2
GarageYrBlt     78
SaleType         1
dtype: int64


In [13]:
# 4.2 NaN «объекта нет» → "None" / 0
print(run_model(config, "ridge_fill_absent"))

(np.float64(0.1447), np.float64(0.0428))


In [25]:
# 4.3 Проверка: порядковые столбцы стали числами
print(X[["KitchenQual", "FireplaceQu", "BsmtExposure", "Functional"]].head())
print(X["KitchenQual"].value_counts())

   KitchenQual  FireplaceQu  BsmtExposure  Functional
0            4            0             1           7
1            3            3             4           7
2            4            3             2           7
3            4            4             1           7
4            4            3             3           7
KitchenQual
3    735
4    586
5    100
2     39
Name: count, dtype: int64


In [26]:
# 4.3 Порядковые категории → числа
print(run_model(config, "ridge_ordinal"))

(np.float64(0.143), np.float64(0.0419))


In [29]:
# 4.4 MSSubClass → категория
print(run_model(config, "ridge_mssubclass_cat"))

(np.float64(0.1421), np.float64(0.0421))


### Выводы 4.1–4.4

| Шаг | Изменение | CV RMSE |
| --- | --- | --- |
| 4.1 | все 79 признаков как есть, Ridge(alpha=10) | 0.1445 ± 0.0425 |
| 4.2 | NaN «объекта нет» → "None" / 0 | 0.1447 ± 0.0428 |
| 4.3 | порядковые категории → числа (без PoolQC) | 0.1430 ± 0.0419 |
| 4.4 | MSSubClass → категория | 0.1421 ± 0.0421 |

- 4.2 по CV ничего не дал: информация «объекта нет» уже была в других столбцах (Fireplaces = 0, GarageCars = 0).
- 4.3: PoolQC убрали. Бассейн есть у 7 домов, после StandardScaler значение уходило на ~17 стандартных отклонений, а выброс 1299 учил модель «бассейн = дешевле». С PoolQC CV был 0.1584.
- 4.4: заполнение LotFrontage медианой по району проверили, эффекта нет (0.1421 → 0.1421). Оставили общую медиану.
- Разброс ± 0.04 почти целиком даёт один фолд с выбросами 524 и 1299, поэтому эффекты небольших изменений пока плохо видны.

In [33]:
# 4.5 Скошенные площади → log_numeric
print(run_model(config, "ridge_log_areas"))

(0.1332, 0.0328)


In [35]:
# 4.6 Без выбросов 524 и 1299
print(X.shape, y.shape)
print(run_model(config, "ridge_drop_outliers"))

(1458, 79) (1458,)
(0.1116, 0.0119)


In [38]:
# 4.7 Без почти константных, дублей и почти нулевых признаков + сабмит
print(run_model(config, "ridge_step4_final"))
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "ridge_preprocessed")

(0.1112, 0.0114)


'outputs\\submission_ridge_preprocessed.csv'

### Выводы 4.5–4.7 и итог предобработки

| Шаг | Изменение | CV RMSE |
| --- | --- | --- |
| 4.5 | 9 скошенных площадей → log1p | 0.1332 ± 0.0328 |
| 4.6 | убрали выбросы 524 и 1299 из train | 0.1116 ± 0.0119 |
| 4.7 | убрали 13 почти константных, дублирующих и почти нулевых признаков | 0.1112 ± 0.0114 |

- 4.5 дал самый большой прирост: цена в логарифме, и площади в логарифме дают связь «процент к проценту». Заодно сжался хвост огромных площадей.
- 4.6: разброс по фолдам упал втрое. Часть прироста CV «бесплатная»: трудные дома ушли из валидации. В test остался похожий дом (Id 2550), поэтому Kaggle хуже CV.
- 4.7: качество то же, признаков 65 вместо 78. При равном качестве выбираем более простую модель.

**Итог шага 4:** Ridge(alpha=10) CV 0.1445 → 0.1112, Kaggle 0.18030 (бейзлайн на 4 признаках) → 0.12569. CV стал примерно на 0.015 оптимистичнее Kaggle.

## Шаг 5. 1 фолд против 5

In [39]:
# 5 Одна модель на всём train против среднего 5 фолдов
_, cv_pred = cross_validate(build_model(config), X, y, X_test, config)

full_model = build_model(config)
full_model.fit(X, y)
full_pred = full_model.predict(X_test)

diff = np.abs(np.expm1(full_pred) - np.expm1(cv_pred)) / np.expm1(cv_pred)
print(f"Средняя разница: {diff.mean():.2%}, максимальная: {diff.max():.2%}")
save_submission(test_ids, full_pred, config, "ridge_full_train")

Средняя разница: 0.25%, максимальная: 2.56%


'outputs\\submission_ridge_full_train.csv'

### Вывод: 1 модель против среднего 5 фолдов

| Вариант | Kaggle |
| --- | --- |
| среднее 5 моделей фолдов | 0.12569 |
| одна модель на всём train | 0.12598 |

Предсказания отличаются в среднем на 0.25% (максимум 2.56%), на Kaggle разница 0.0003, то есть на практике одинаково. Ridge стабилен, лишние 20% данных почти не меняют веса. Оставляем среднее 5 фолдов: CV и сабмит получаются из одного прогона.

## Шаг 6. Классические модели

In [40]:
# 6.1 LinearRegression без штрафа
cfg = OmegaConf.merge(config, {"model": {"name": "linreg"}})
cfg.model.params = {}
print("linreg", run_model(cfg))

linreg (0.1164, 0.011)


In [41]:
# 6.1 Ridge: сетка по alpha
for alpha in [1, 3, 10, 30, 100]:
    cfg = OmegaConf.merge(config, {"model": {"name": "ridge"}})
    cfg.model.params = {"alpha": alpha}
    print("ridge", alpha, run_model(cfg))

ridge 1 (0.1134, 0.0112)
ridge 3 (0.112, 0.0112)
ridge 10 (0.1112, 0.0114)
ridge 30 (0.1117, 0.0119)
ridge 100 (0.1143, 0.0122)


In [42]:
# 6.1 Lasso: сетка по alpha
for alpha in [0.0001, 0.0003, 0.001, 0.003, 0.01]:
    cfg = OmegaConf.merge(config, {"model": {"name": "lasso"}})
    cfg.model.params = {"alpha": alpha, "max_iter": 20000}
    print("lasso", alpha, run_model(cfg))

lasso 0.0001 (0.1124, 0.0122)
lasso 0.0003 (0.1108, 0.0126)
lasso 0.001 (0.1122, 0.0123)
lasso 0.003 (0.118, 0.0132)
lasso 0.01 (0.1244, 0.0131)


In [43]:
# 6.1 ElasticNet: сетка по alpha и l1_ratio
for alpha in [0.0005, 0.001, 0.003]:
    for l1_ratio in [0.3, 0.5]:
        cfg = OmegaConf.merge(config, {"model": {"name": "elasticnet"}})
        cfg.model.params = {"alpha": alpha, "l1_ratio": l1_ratio, "max_iter": 20000}
        print("elasticnet", alpha, l1_ratio, run_model(cfg))

elasticnet 0.0005 0.3 (0.1115, 0.0122)
elasticnet 0.0005 0.5 (0.1109, 0.0124)
elasticnet 0.001 0.3 (0.1108, 0.0122)
elasticnet 0.001 0.5 (0.1108, 0.0124)
elasticnet 0.003 0.3 (0.1123, 0.0122)
elasticnet 0.003 0.5 (0.1148, 0.0125)


In [44]:
# 6.2 KNN: сетка по числу соседей и весам
for k in [3, 5, 10, 20]:
    for weights in ["uniform", "distance"]:
        cfg = OmegaConf.merge(config, {"model": {"name": "knn"}})
        cfg.model.params = {"n_neighbors": k, "weights": weights}
        print("knn", k, weights, run_model(cfg))

knn 3 uniform (0.1686, 0.0093)
knn 3 distance (0.1679, 0.0097)
knn 5 uniform (0.1631, 0.0101)
knn 5 distance (0.1621, 0.0104)
knn 10 uniform (0.1652, 0.0096)
knn 10 distance (0.1637, 0.0099)
knn 20 uniform (0.1708, 0.0082)
knn 20 distance (0.1688, 0.0084)


In [45]:
# 6.3 Дерево: сетка по глубине и минимальному листу
for max_depth in [3, 5, 7, 10, None]:
    for min_samples_leaf in [1, 5, 10]:
        cfg = OmegaConf.merge(config, {"model": {"name": "tree"}})
        cfg.model.params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf,
                            "random_state": "${seed}"}
        print("tree", max_depth, min_samples_leaf, run_model(cfg))

tree 3 1 (0.2283, 0.0102)
tree 3 5 (0.2283, 0.0102)
tree 3 10 (0.2283, 0.0102)
tree 5 1 (0.1963, 0.0098)
tree 5 5 (0.1957, 0.0098)
tree 5 10 (0.1953, 0.0145)
tree 7 1 (0.1965, 0.0122)
tree 7 5 (0.1879, 0.0177)
tree 7 10 (0.1841, 0.0155)
tree 10 1 (0.2063, 0.0104)
tree 10 5 (0.1873, 0.018)
tree 10 10 (0.1836, 0.0139)
tree None 1 (0.2049, 0.0091)
tree None 5 (0.1882, 0.017)
tree None 10 (0.1838, 0.0138)


In [46]:
# 6.4 Случайный лес: сетка по доле признаков и минимальному листу
for max_features in [0.3, 0.5, 1.0]:
    for min_samples_leaf in [1, 3]:
        cfg = OmegaConf.merge(config, {"model": {"name": "rf"}})
        cfg.model.params = {"n_estimators": 300, "max_features": max_features,
                            "min_samples_leaf": min_samples_leaf,
                            "n_jobs": -1, "random_state": "${seed}"}
        print("rf", max_features, min_samples_leaf, run_model(cfg))

rf 0.3 1 (0.1323, 0.0147)
rf 0.3 3 (0.134, 0.0144)
rf 0.5 1 (0.1332, 0.0153)
rf 0.5 3 (0.1349, 0.0153)
rf 1.0 1 (0.1372, 0.0163)
rf 1.0 3 (0.1378, 0.0161)


In [47]:
# 6.5 LightGBM: сетка по learning_rate и числу листьев
for lr, n in [(0.05, 1000), (0.03, 2000)]:
    for num_leaves in [8, 15]:
        cfg = OmegaConf.merge(config, {"model": {"name": "lgbm"}})
        cfg.model.params = {"n_estimators": n, "learning_rate": lr, "num_leaves": num_leaves,
                            "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.5,
                            "random_state": "${seed}", "verbose": -1}
        print("lgbm", lr, num_leaves, run_model(cfg))

lgbm 0.05 8 (0.121, 0.0149)
lgbm 0.05 15 (0.1228, 0.0155)
lgbm 0.03 8 (0.1196, 0.0154)
lgbm 0.03 15 (0.1218, 0.0151)


In [48]:
# 6.5 XGBoost: сетка по learning_rate и глубине
for lr, n in [(0.05, 1000), (0.03, 2000)]:
    for max_depth in [3, 4]:
        cfg = OmegaConf.merge(config, {"model": {"name": "xgb"}})
        cfg.model.params = {"n_estimators": n, "learning_rate": lr, "max_depth": max_depth,
                            "subsample": 0.8, "colsample_bytree": 0.5,
                            "random_state": "${seed}"}
        print("xgb", lr, max_depth, run_model(cfg))

xgb 0.05 3 (0.1141, 0.015)
xgb 0.05 4 (0.1169, 0.0154)
xgb 0.03 3 (0.1137, 0.0154)
xgb 0.03 4 (0.1159, 0.016)


In [49]:
# 6.5 CatBoost: сетка по learning_rate и глубине
for lr, n in [(0.05, 1000), (0.03, 2000)]:
    for depth in [4, 6]:
        cfg = OmegaConf.merge(config, {"model": {"name": "catboost"}})
        cfg.model.params = {"iterations": n, "learning_rate": lr, "depth": depth,
                            "random_state": "${seed}", "verbose": 0}
        print("catboost", lr, depth, run_model(cfg))

catboost 0.05 4 (0.1127, 0.0147)
catboost 0.05 6 (0.1147, 0.0152)
catboost 0.03 4 (0.1128, 0.015)
catboost 0.03 6 (0.1147, 0.0152)


In [51]:
# 6.6 Лучшая модель после ручных сеток: Lasso(alpha=0.0003), сабмит
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "lasso")

'outputs\\submission_lasso.csv'

### Выводы шага 6

| Модель | Лучшие параметры | CV RMSE |
| --- | --- | --- |
| Lasso | alpha=0.0003 | 0.1108 ± 0.0126 |
| ElasticNet | alpha=0.001, l1_ratio=0.5 | 0.1108 ± 0.0124 |
| Ridge | alpha=10 | 0.1112 ± 0.0114 |
| CatBoost | lr 0.05, depth 4 | 0.1127 ± 0.0147 |
| XGBoost | lr 0.03, depth 3 | 0.1137 ± 0.0154 |
| LinearRegression | | 0.1164 ± 0.011 |
| LightGBM | lr 0.03, 8 листьев | 0.1196 ± 0.0154 |
| Случайный лес | max_features 0.3, лист 1 | 0.1323 ± 0.0147 |
| KNN | k=5, distance | 0.1621 ± 0.0104 |
| Дерево | depth 10, лист 10 | 0.1836 ± 0.0139 |

- Линейные модели с регуляризацией и бустинги на равных, разница в пределах разброса. В Титанике лучшим был лес, здесь линейные: цена плавно зависит от площади и качества.
- Регуляризация даёт +0.005 к обычной регрессии. По alpha видна «яма»: слева переобучение, справа недообучение.
- Деревья: одиночное плохо (ступенчатые предсказания), лес лучше за счёт усреднения, бустинг ещё лучше за счёт последовательного исправления ошибок. Мелкие деревья (depth 3–4) в бустинге лучше глубоких.
- Lasso обнулил 100 из 209 столбцов, то есть сам отобрал признаки.
- В конфиг идёт Lasso(alpha=0.0003): лучший по CV и проще ElasticNet. Kaggle 0.12797 против 0.12569 у Ridge: разница в пределах шума публичного лидерборда, решаем по CV.